# Taller Práctico 2 — LogiAndes S.A.
## Reto: Convertir datos en decisiones monitoreables

**Visualización, Storytelling y Business Intelligence** · **Grupo 5**

Mayra Sanchez · Noemí Fernandez · Christian Pazmiño

| Entregable | Archivo |
|---|---|
| Notebook con el procesamiento, los KPIs y las respuestas | `Taller_2_Grupo5.ipynb` (este documento) |
| Capa semántica compartida (datos, KPIs y gráficos) | `kpis_grupo5.py` |
| Prototipo de dashboard en Streamlit | `app.py` → `streamlit run app.py` |
| KPIs exportados para verificar la coherencia con el dashboard | `outputs/` |

**Continuidad con el Taller 1.** En el Taller 1 construimos gráficos aislados: barras de ventas, boxplots de entrega, una dispersión entre pedidos y reclamos, y una vista interactiva de tasas. En este taller esos análisis se convierten en **cinco KPIs con ficha técnica, meta y semáforo**, y luego en un **dashboard integrado** organizado por audiencias.

**Estructura del cuaderno**
1. Problema empresarial y necesidades de las audiencias
2. Preparación de los datos y control de calidad
3. KPIs: ficha técnica, cálculo, visualización y resultado (KPI 1 a KPI 5)
4. Estructura del dashboard (vista ejecutiva, análisis territorial y detalle operativo)
5. Hallazgos y respuestas para la toma de decisiones
6. Coherencia entre el Notebook y el dashboard, y trazabilidad
7. Reflexión: sobreinteractividad, sesgos visuales, privacidad, gobernanza y límites interpretativos
8. Conclusiones y ejecución

# 1. Problema empresarial y necesidades de las audiencias

Durante el primer trimestre de 2026, LogiAndes S.A. recibió **señales contradictorias**. El área comercial reporta que las ventas crecen, especialmente en Guayas y Pichincha. Atención al cliente reporta más reclamos en algunas zonas. La gerencia necesita saber si el aumento de reclamos es un efecto normal del mayor volumen, una diferencia real en la calidad del servicio entre territorios o un problema de ciertos canales logísticos.

En el Taller 1 respondimos con una lectura inicial. Ahora la empresa necesita **monitorear** estas señales de forma continua, con indicadores definidos de manera única, metas explícitas y una herramienta que cada área pueda consultar sin depender del analista.

### Audiencias y sus necesidades

| Audiencia | Pregunta de negocio | Decisiones que apoya | KPIs principales | Sección del dashboard |
|---|---|---|---|---|
| **Gerencia general** | ¿Crecemos sin deteriorar el servicio? ¿Qué provincias intervenir primero? | Priorizar territorios y asignar recursos | Los 5 KPIs (tablero de semáforos) | Vista ejecutiva |
| **Área comercial** | ¿Qué provincias cumplen la meta de ventas y crecen en valor por pedido? | Metas comerciales, mezcla de clientes y canales | KPI 1 Meta de ventas, KPI 2 Ticket promedio | Vista ejecutiva y Análisis territorial |
| **Operaciones** | ¿Dónde se incumple el plazo y dónde las entregas son lentas o irregulares? | Rutas, capacidad de reparto, cobertura | KPI 4 Mediana de entrega, KPI 5 Entregas a tiempo | Detalle operativo |
| **Finanzas** | ¿El ingreso crece de forma sostenida? ¿Crece por más pedidos o por pedidos de mayor valor? | Proyección de ingresos, evaluación de canales | KPI 1, KPI 2 | Vista ejecutiva |
| **Atención al cliente** | ¿Qué provincias y motivos concentran los reclamos? ¿Se relacionan con las entregas tardías? | Protocolos, comunicación proactiva | KPI 3 Tasa de reclamos, KPI 5 | Detalle operativo |

El dashboard tiene un selector **“Vista para”** que muestra primero los KPIs de cada audiencia y su pregunta clave, sin crear cinco dashboards distintos. Todas las audiencias consultan una sola fuente de verdad.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.io as pio

import kpis_grupo5 as kg   # capa semántica compartida con el dashboard (app.py)

pio.renderers.default = "plotly_mimetype+notebook_connected"   # VS Code / JupyterLab y respaldo HTML
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)   # muestra el texto completo de cada celda
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

def tabla(df):
    # Tabla con texto alineado a la izquierda y salto de línea (legible en el Notebook)
    return df.style.set_properties(**{"text-align": "left", "white-space": "normal"}).format(precision=1)

audiencias = pd.DataFrame(kg.AUDIENCIAS).T
audiencias["kpis"] = audiencias["kpis"].apply(lambda ks: ", ".join(kg.KPIS[k]["codigo"] for k in ks))
tabla(audiencias.rename(columns={"pregunta": "Pregunta clave", "kpis": "KPIs", "seccion": "Sección"}))

,Pregunta clave,KPIs,Sección
Gerencia general,¿Crecemos sin deteriorar el servicio? ¿Qué provincias requieren intervención primero?,"KPI 1, KPI 3, KPI 5, KPI 4, KPI 2",Vista ejecutiva
Área comercial,"¿Qué provincias cumplen la meta de ventas y crecen en valor por pedido, no solo en volumen?","KPI 1, KPI 2",Vista ejecutiva y Análisis territorial
Operaciones,¿Dónde se incumple el plazo de entrega y dónde las entregas son lentas o irregulares?,"KPI 4, KPI 5",Detalle operativo
Finanzas,¿El ingreso crece de forma sostenida y por qué vía: más pedidos o pedidos de mayor valor?,"KPI 1, KPI 2",Vista ejecutiva
Atención al cliente,"¿Qué provincias y motivos concentran los reclamos, y cuánto se relacionan con entregas tardías?","KPI 3, KPI 5",Detalle operativo


# 2. Preparación de los datos y control de calidad

**Fuente:** base de pedidos de LogiAndes S.A. (`data/logiandes.csv`), entregada por ingeniería de datos. **Unidad de análisis:** el pedido. **Periodo:** 1 de enero a 31 de marzo de 2026.

La función `kg.cargar_datos()` no modifica los valores originales. Solo agrega las **variables derivadas** que necesitan los KPIs:

| Variable derivada | Regla | Uso |
|---|---|---|
| `mes_num`, `mes` | Mes de `fecha_pedido` | Frecuencia mensual de los KPIs |
| `semana`, `hora`, `dia_semana` | A partir de `fecha_pedido` | Seguimiento semanal y demanda por hora |
| `a_tiempo` | 1 si `tiempo_individual_entrega_min` ≤ 180 | KPI 5 |
| `con_reclamo` | `reclamos_registrados` > 0 | KPI 3 |
| `tramo_distancia` | 0–5, 5–10, 10–15, 15–20, > 20 km | Separar el efecto de la distancia |
| `motivo_reclamo` | Vacíos → “Sin reclamo” | Evitar nulos ambiguos |

In [2]:
df = kg.cargar_datos()
print(f"Registros: {len(df):,} pedidos | Periodo: {df['fecha_pedido'].min():%d-%m-%Y} a {df['fecha_pedido'].max():%d-%m-%Y}")
df.head()

Registros: 5,798 pedidos | Periodo: 01-01-2026 a 31-03-2026


,pedido_id,fecha_pedido,provincia,region,canal_entrega,tipo_cliente,tiempo_atencion_min,tiempo_individual_entrega_min,reclamos_registrados,motivo_reclamo,ventas_asociadas_usd,distancia_km,fecha,mes_num,mes,semana,hora,dia_semana,a_tiempo,con_reclamo,tramo_distancia
0,LA-004235,2026-01-01 07:16:00,Manabí,Costa,Programado,Comercio minorista,8.90,212.10,0,Sin reclamo,16.85,32.50,2026-01-01,1,Enero,2025-12-29,7,Jue,0,False,> 20 km
1,LA-002169,2026-01-01 07:33:00,Pichincha,Sierra,Programado,Comercio minorista,5.90,136.00,0,Sin reclamo,17.11,9.90,2026-01-01,1,Enero,2025-12-29,7,Jue,1,False,5–10 km
2,LA-000024,2026-01-01 07:35:00,Guayas,Costa,Estándar urbano,Comercio minorista,7.90,150.30,0,Sin reclamo,17.04,8.90,2026-01-01,1,Enero,2025-12-29,7,Jue,1,False,5–10 km
3,LA-000178,2026-01-01 07:44:00,Guayas,Costa,Estándar urbano,Comercio minorista,5.40,118.40,0,Sin reclamo,17.54,4.80,2026-01-01,1,Enero,2025-12-29,7,Jue,1,False,0–5 km
4,LA-004810,2026-01-01 07:59:00,Tungurahua,Sierra,Estándar urbano,Comercio minorista,9.90,135.90,0,Sin reclamo,13.95,5.20,2026-01-01,1,Enero,2025-12-29,7,Jue,1,False,5–10 km


In [3]:
# Diccionario de datos
diccionario = pd.DataFrame({
    "Variable": ["pedido_id", "fecha_pedido", "provincia", "region", "canal_entrega", "tipo_cliente",
                 "tiempo_atencion_min", "tiempo_individual_entrega_min", "reclamos_registrados",
                 "motivo_reclamo", "ventas_asociadas_usd", "distancia_km"],
    "Tipo": ["Identificador (seudónimo)", "Temporal", "Categórica", "Categórica", "Categórica", "Categórica",
             "Numérica continua", "Numérica continua", "Binaria (0/1)", "Categórica", "Numérica continua",
             "Numérica continua"],
    "Uso en la solución": ["Conteo de pedidos y detalle", "Mes, semana y hora", "Análisis territorial", "Agrupación",
                           "Comparación de canales", "Segmentación comercial", "Detalle operativo",
                           "KPI 4 y KPI 5", "KPI 3", "Pareto de motivos", "KPI 1 y KPI 2", "Control de distancia"],
})
diccionario["Nulos"] = diccionario["Variable"].map(pd.read_csv(kg.RUTA_DATOS, encoding="utf-8-sig").isna().sum())
tabla(diccionario)

,Variable,Tipo,Uso en la solución,Nulos
0,pedido_id,Identificador (seudónimo),Conteo de pedidos y detalle,0
1,fecha_pedido,Temporal,"Mes, semana y hora",0
2,provincia,Categórica,Análisis territorial,0
3,region,Categórica,Agrupación,0
4,canal_entrega,Categórica,Comparación de canales,0
5,tipo_cliente,Categórica,Segmentación comercial,0
6,tiempo_atencion_min,Numérica continua,Detalle operativo,0
7,tiempo_individual_entrega_min,Numérica continua,KPI 4 y KPI 5,0
8,reclamos_registrados,Binaria (0/1),KPI 3,0
9,motivo_reclamo,Categórica,Pareto de motivos,5611


In [4]:
calidad = kg.calidad_datos(df)
print("Controles superados:", calidad["Cumple"].sum(), "de", len(calidad))
tabla(calidad)

Controles superados: 8 de 8


,Control,Cumple,Detalle
0,Identificador de pedido único,True,"5,798 ids únicos"
1,Sin pedidos duplicados,True,0 duplicados
2,Fechas dentro de enero–marzo 2026,True,01-01-2026 a 31-03-2026
3,Todos los días del trimestre tienen pedidos,True,90 de 90 días
4,"Tiempos, distancias y ventas positivos",True,mín > 0
5,Reclamos binarios (0/1),True,"valores {0, 1}"
6,Todo reclamo tiene motivo y viceversa,True,187 reclamos con motivo
7,Región coherente con provincia,True,1 región por provincia


In [5]:
df[["tiempo_atencion_min", "tiempo_individual_entrega_min", "distancia_km", "ventas_asociadas_usd"]].describe(
    percentiles=[.25, .5, .75, .9]).T

,count,mean,std,min,25%,50%,75%,90%,max
tiempo_atencion_min,"5,798.00",7.96,2.00,2.00,6.60,7.90,9.30,10.50,15.60
tiempo_individual_entrega_min,"5,798.00",132.40,52.77,28.00,100.70,129.30,157.30,189.53,520.00
distancia_km,"5,798.00",9.82,7.19,1.00,4.70,8.00,13.10,19.20,45.00
ventas_asociadas_usd,"5,798.00",24.58,10.74,6.50,16.91,22.24,29.78,38.75,95.00


**Lectura.** La base supera los 8 controles de calidad, así que no requiere limpieza: solo se derivan variables. Los 5.611 nulos de `motivo_reclamo` corresponden a pedidos **sin** reclamo, es decir, son estructurales. Los 90 días del trimestre tienen pedidos, lo que permite calcular ventas por día sin huecos. El tiempo de entrega tiene una cola larga (máximo 520 min frente a una mediana de 129 min). Por eso el KPI 4 usa la mediana y no el promedio.

# 3. KPIs: ficha técnica, cálculo y resultado

Se definen **cinco KPIs**, cada uno con su ficha técnica: nombre, objetivo, fórmula, fuente, frecuencia, meta, umbral, interpretación y responsable. Todos se calculan con `kpis_grupo5.py`, que también usa el dashboard.

| KPI | Indicador | Sentido | Meta (🟢) | Crítico (🔴) | Audiencia principal |
|---|---|---|---|---|---|
| KPI 1 | Cumplimiento de meta de ventas mensuales | Mayor es mejor | ≥ 95 % | < 80 % | Comercial, finanzas |
| KPI 2 | Ticket promedio por pedido (índice vs. enero) | Mayor es mejor | ≥ 100 % | < 95 % | Comercial, finanzas |
| KPI 3 | Tasa de reclamos | Menor es mejor | ≤ 3 % | > 5 % | Atención al cliente |
| KPI 4 | Mediana del tiempo de entrega | Menor es mejor | ≤ 120 min | > 150 min | Operaciones |
| KPI 5 | Porcentaje de entregas a tiempo (≤ 180 min) | Mayor es mejor | ≥ 90 % | < 80 % | Operaciones, atención al cliente |

**Criterios comunes:**
* **Semáforo con texto y color** (🟢 Cumple, 🟡 Atención, 🔴 Crítico), para no depender solo del color.
* **KPI 1 y KPI 2** comparan cada provincia con **su propio enero** (línea base), porque su escala y su mezcla de clientes son distintas. Enero no se evalúa.
* **KPI 3 y KPI 5** son proporciones: se reporta su **margen de error** (intervalo de confianza de Wilson al 95 %).
* **Las metas son propuestas con fines académicos** y deben validarse con cada responsable (sección 7.4).
* En cada gráfico, el **total de la empresa** aparece en azul. En las provincias, **febrero** aparece en gris y **marzo** con el color de su semáforo.

In [6]:
tabla(kg.ficha_tecnica())

,Nombre,Objetivo,Fórmula,Fuente (columnas),Frecuencia,Meta,Umbral,Interpretación,Responsable
KPI,,,,,,,,,
KPI 1,Cumplimiento de meta de ventas mensuales,"Evaluar si las ventas de cada mes alcanzan la meta de crecimiento definida, a nivel empresa y por provincia.","Ventas del mes (Σ ventas_asociadas_usd) ÷ Meta del mes × 100. Meta del mes = venta diaria de enero × 1,05 × días del mes (enero = línea base).","ventas_asociadas_usd, fecha_pedido, provincia",Mensual.,"100 %: crecer 5 % sobre la venta diaria de enero (meta propuesta con fines académicos, a validar con gerencia).","🟢 ≥ 95 % | 🟡 80 % – 94,9 % | 🔴 < 80 %","Mayor es mejor. Bajo 95 % indica rezago frente a la meta. Enero no se evalúa (línea base). La meta se prorratea por días porque febrero tiene 28 días y marzo 31. Mide volumen de ingreso, no rentabilidad (la base no incluye costos).",Gerencia comercial
KPI 2,Ticket promedio por pedido,"Detectar si el ingreso crece por más pedidos o por pedidos de mayor valor, a nivel empresa y por provincia.",Ticket = Σ ventas_asociadas_usd ÷ N.º de pedidos del mes. Índice = Ticket del mes ÷ Ticket de enero × 100.,"ventas_asociadas_usd, pedido_id, fecha_pedido, provincia",Mensual.,Mantener o superar el ticket de enero (índice ≥ 100 %). Cada provincia se compara con su propio enero.,"🟢 ≥ 100 % | 🟡 95 % – 99,9 % | 🔴 < 95 %",Mayor es mejor. Bajo 100 % el valor por pedido cae frente a la línea base. Se complementa con la mediana porque el promedio es sensible a pedidos de alto valor. No mide rentabilidad.,Área comercial y finanzas
KPI 3,Tasa de reclamos,"Monitorear la calidad del servicio desde la experiencia del cliente, sin el efecto del volumen de pedidos.","Σ reclamos_registrados ÷ N.º de pedidos × 100, con intervalo de confianza de Wilson al 95 %.","reclamos_registrados, motivo_reclamo, pedido_id, fecha_pedido, provincia","Mensual, con lectura acumulada trimestral para confirmar tendencias.","≤ 3 reclamos por cada 100 pedidos, igual para todas las provincias (estándar de servicio).",🟢 ≤ 3 % | 🟡 3 % – 5 % | 🔴 > 5 %,Menor es mejor. Sobre 5 % indica un problema de servicio. Con pocos reclamos por mes la tasa varía por azar: confirmar con el intervalo de confianza y con la tasa trimestral antes de actuar.,"Atención al cliente, con operaciones"
KPI 4,Mediana del tiempo de entrega,Monitorear la rapidez de la entrega por provincia y mes frente al estándar operativo.,Mediana de tiempo_individual_entrega_min del mes. Complemento: RIC = Q3 − Q1 (variabilidad).,"tiempo_individual_entrega_min, canal_entrega, fecha_pedido, provincia",Mensual (seguimiento semanal en operaciones).,"Mediana ≤ 120 minutos, nivel que ya alcanzan Azuay y Tungurahua (exigente pero alcanzable).",🟢 ≤ 120 min | 🟡 120 – 150 min | 🔴 > 150 min,Menor es mejor. Se usa la mediana porque los tiempos tienen casos extremos. Sobre 150 min indica un problema operativo. El RIC muestra si las entregas son regulares o impredecibles.,Operaciones y logística
KPI 5,Porcentaje de entregas a tiempo,Medir qué proporción de pedidos llega dentro del plazo máximo aceptable: lo que percibe cada cliente.,Pedidos con tiempo_individual_entrega_min ≤ 180 ÷ Total de pedidos del mes × 100 (IC de Wilson 95 %).,"tiempo_individual_entrega_min, pedido_id, fecha_pedido, provincia",Mensual (seguimiento semanal en operaciones).,≥ 90 % de pedidos en 180 min o menos (2 h = tiempo ideal del KPI 4; 3 h = máximo aceptable).,"🟢 ≥ 90 % | 🟡 80 % – 89,9 % | 🔴 < 80 %",Mayor es mejor. Complementa al KPI 4: la mediana muestra el tiempo típico y este KPI cuántos pedidos se salen del plazo. El plazo es único para todos los canales; Programado podría requerir otra ventana.,"Operaciones, con seguimiento de atención al cliente"


## 3.1 Indicador 1: Cumplimiento de meta de ventas mensuales

In [7]:
kg.mostrar_ficha("k1")

Campo,Descripción
Nombre,KPI 1 – Cumplimiento de meta de ventas mensuales
Objetivo,"Evaluar si las ventas de cada mes alcanzan la meta de crecimiento definida, a nivel empresa y por provincia."
Fórmula,"Ventas del mes (Σ ventas_asociadas_usd) ÷ Meta del mes × 100. Meta del mes = venta diaria de enero × 1,05 × días del mes (enero = línea base)."
Fuente,"Base de pedidos LogiAndes S.A. (logiandes.csv), enero–marzo 2026: columnas ventas_asociadas_usd, fecha_pedido, provincia."
Frecuencia,Mensual.
Meta,"100 %: crecer 5 % sobre la venta diaria de enero (meta propuesta con fines académicos, a validar con gerencia)."
Umbral,"🟢 ≥ 95 % | 🟡 80 % – 94,9 % | 🔴 < 80 %"
Interpretación,"Mayor es mejor. Bajo 95 % indica rezago frente a la meta. Enero no se evalúa (línea base). La meta se prorratea por días porque febrero tiene 28 días y marzo 31. Mide volumen de ingreso, no rentabilidad (la base no incluye costos)."
Responsable,Gerencia comercial


### Mejora metodológica: ajuste por días del mes

En la primera versión, la meta mensual era la misma para todos los meses (ventas de enero × 1,05). Pero **febrero tiene 28 días y marzo 31**, así que febrero quedaba penalizado y marzo favorecido sin que cambiara el desempeño. Por eso la meta ahora se **prorratea por días**:

> **Meta del mes = venta diaria de enero × 1,05 × días del mes**

La tabla compara ambas versiones. El parámetro `kg.AJUSTE_CALENDARIO` permite volver a la fórmula original.

In [8]:
kg.AJUSTE_CALENDARIO = False
original = kg.tabla_mensual(df).set_index("mes")[["ventas_usd", "meta_usd", "k1", "estado_k1"]]
kg.AJUSTE_CALENDARIO = True
ajustada = kg.tabla_mensual(df).set_index("mes")[["dias", "ventas_dia", "meta_usd", "k1", "estado_k1"]]

comparacion = pd.concat({"Meta fija (versión anterior)": original, "Meta ajustada por días": ajustada}, axis=1)
comparacion.round(1)

Meta fija (versión anterior)                             \
                          ventas_usd  meta_usd     k1 estado_k1   
mes                                                               
Enero                      44,955.90 47,203.70    NaN      base   
Febrero                    46,300.40 47,203.70  98.10     verde   
Marzo                      51,286.40 47,203.70 108.60     verde   

        Meta ajustada por días                                        
                          dias ventas_dia  meta_usd     k1 estado_k1  
mes                                                                   
Enero                       31   1,450.20 47,203.70    NaN      base  
Febrero                     28   1,653.60 42,635.60 108.60     verde  
Marzo                       31   1,654.40 47,203.70 108.60     verde

In [9]:
kg.fig_ventas_por_dia(df).show()

**Lectura.** Por día, las ventas subieron **14 % en febrero** (de USD 1.450 a USD 1.654) y se **mantuvieron en marzo** (USD 1.654). El “+10,8 % de marzo” de la versión anterior era un **efecto calendario**: marzo tiene tres días más que febrero. Con la meta ajustada, la empresa cumple **108,6 % en febrero y en marzo**. Es la misma conclusión (verde), pero mejor fundamentada: el crecimiento ocurrió en febrero y luego se sostuvo.

In [10]:
# Cumplimiento por mes (empresa) con la meta ajustada
ventas_mes = kg.tabla_mensual(df)
ventas_mes[["mes", "dias", "ventas_usd", "ventas_dia", "meta_usd", "k1", "variacion_ventas_pct",
            "variacion_ventas_dia_pct", "estado_k1"]].round(1)

,mes,dias,ventas_usd,ventas_dia,meta_usd,k1,variacion_ventas_pct,variacion_ventas_dia_pct,estado_k1
0,Enero,31,"44,955.90","1,450.20","47,203.70",NaN,NaN,NaN,base
1,Febrero,28,"46,300.40","1,653.60","42,635.60",108.60,3.00,14.00,verde
2,Marzo,31,"51,286.40","1,654.40","47,203.70",108.60,10.80,0.00,verde


In [11]:
# Cumplimiento por provincia: cada una frente a su propio enero
prov_mes = kg.tabla_mensual(df, "provincia")
kpi1_prov = prov_mes[prov_mes["mes_num"] > 1].pivot(index="provincia", columns="mes", values="k1")[["Febrero", "Marzo"]]
kpi1_prov["Ventas por día (ene → feb → mar)"] = (prov_mes.pivot(index="provincia", columns="mes_num", values="ventas_dia")
                                                 .apply(lambda r: " → ".join(f"${v:,.0f}" for v in r), axis=1))
kpi1_prov["Estado marzo"] = kpi1_prov["Marzo"].apply(lambda v: kg.etiqueta_estado(kg.estado("k1", v)))
kpi1_prov.sort_values("Marzo").round(1)

mes,Febrero,Marzo,Ventas por día (ene → feb → mar),Estado marzo
provincia,,,,
Loja,102.80,87.90,$103 → $111 → $95,🟡 Atención
Tungurahua,108.00,103.40,$145 → $164 → $157,🟢 Cumple
Guayas,110.50,109.00,$461 → $535 → $527,🟢 Cumple
Manabí,104.60,109.10,$166 → $182 → $190,🟢 Cumple
Azuay,103.60,109.90,$183 → $200 → $212,🟢 Cumple
Pichincha,112.10,114.80,$392 → $462 → $473,🟢 Cumple


In [12]:
kg.fig_kpi(df, "k1", mes=3, leyenda_abajo=False,
           titulo="<b>La empresa supera la meta de ventas en febrero y marzo (109 %); solo Loja queda bajo la meta en marzo</b>").show()

**Resultado del KPI 1.**
- **A nivel empresa, el KPI está en verde los dos meses evaluados:** 108,6 % en febrero y 108,6 % en marzo. Las ventas por día crecieron 14 % entre enero y febrero y luego se mantuvieron estables.
- **Por provincia,** todas cumplen la meta en febrero. En marzo todas siguen en verde excepto **Loja**, que pasa a amarillo (87,9 %).
- **Corrección frente a la versión anterior:** Loja no cae de forma sostenida. Por día vendió USD 103 en enero, USD 111 en febrero y USD 95 en marzo. Su caída es **de marzo** y debe confirmarse en abril antes de calificarla como tendencia.

**Implicación.** El crecimiento general es positivo, pero conviene vigilar a Loja. Antes de decidir acciones, hay que revisar si su caída de marzo se debe a menos pedidos, a un menor ticket (KPI 2) o a los problemas de servicio que muestran el KPI 4 y el KPI 5.

**Limitación.** Con tres meses de datos, la línea base de enero puede estar influida por la estacionalidad: enero suele ser un mes de baja actividad. La meta debe validarse con la gerencia y recalcularse cuando haya más historial.

## 3.2 Indicador 2: Ticket promedio por pedido

In [13]:
kg.mostrar_ficha("k2")

Campo,Descripción
Nombre,KPI 2 – Ticket promedio por pedido
Objetivo,"Detectar si el ingreso crece por más pedidos o por pedidos de mayor valor, a nivel empresa y por provincia."
Fórmula,Ticket = Σ ventas_asociadas_usd ÷ N.º de pedidos del mes. Índice = Ticket del mes ÷ Ticket de enero × 100.
Fuente,"Base de pedidos LogiAndes S.A. (logiandes.csv), enero–marzo 2026: columnas ventas_asociadas_usd, pedido_id, fecha_pedido, provincia."
Frecuencia,Mensual.
Meta,Mantener o superar el ticket de enero (índice ≥ 100 %). Cada provincia se compara con su propio enero.
Umbral,"🟢 ≥ 100 % | 🟡 95 % – 99,9 % | 🔴 < 95 %"
Interpretación,Mayor es mejor. Bajo 100 % el valor por pedido cae frente a la línea base. Se complementa con la mediana porque el promedio es sensible a pedidos de alto valor. No mide rentabilidad.
Responsable,Área comercial y finanzas


In [14]:
ticket_mes = kg.tabla_mensual(df)
print(f"Meta (ticket de enero): ${ticket_mes.loc[0, 'ticket_usd']:.2f}")
ticket_mes[["mes", "ventas_usd", "pedidos", "mediana_usd", "ticket_usd", "meta_ticket", "k2", "estado_k2"]].round(2)

Meta (ticket de enero): $24.47


,mes,ventas_usd,pedidos,mediana_usd,ticket_usd,meta_ticket,k2,estado_k2
0,Enero,"44,955.89",1837,21.77,24.47,24.47,NaN,base
1,Febrero,"46,300.45",1899,22.05,24.38,24.47,99.63,amarillo
2,Marzo,"51,286.39",2062,22.92,24.87,24.47,101.63,verde


In [15]:
kpi2_prov = prov_mes[prov_mes["mes_num"] > 1].pivot(index="provincia", columns="mes", values="k2")[["Febrero", "Marzo"]]
kpi2_prov["Ticket marzo (USD)"] = prov_mes[prov_mes["mes_num"] == 3].set_index("provincia")["ticket_usd"]
kpi2_prov["Mediana marzo (USD)"] = prov_mes[prov_mes["mes_num"] == 3].set_index("provincia")["mediana_usd"]
kpi2_prov["Estado marzo"] = kpi2_prov["Marzo"].apply(lambda v: kg.etiqueta_estado(kg.estado("k2", v)))
kpi2_prov.sort_values("Marzo").round(1)

mes,Febrero,Marzo,Ticket marzo (USD),Mediana marzo (USD),Estado marzo
provincia,,,,,
Loja,96.80,96.80,20.40,18.90,🟡 Atención
Guayas,98.90,98.30,26.40,24.70,🟡 Atención
Manabí,100.10,98.80,22.60,20.80,🟡 Atención
Pichincha,102.80,103.10,26.90,24.80,🟢 Cumple
Tungurahua,94.50,104.90,23.30,21.50,🟢 Cumple
Azuay,99.00,106.10,23.10,20.70,🟢 Cumple


In [16]:
kg.fig_kpi(df, "k2", mes=3, leyenda_abajo=False,
           titulo="<b>El ticket de la empresa sube levemente en marzo; Guayas, Manabí y Loja quedan bajo su nivel de enero</b>").show()

In [17]:
# Ticket por tipo de cliente y canal (para la recomendación comercial)
kg.tabla_periodo(df, "tipo_cliente")[["pedidos", "ticket_usd"]].join(
    kg.tabla_periodo(df, "canal_entrega")[["pedidos", "ticket_usd"]], how="outer", lsuffix=" (cliente)", rsuffix=" (canal)").round(2)

,pedidos (cliente),ticket_usd (cliente),pedidos (canal),ticket_usd (canal)
tipo_cliente,,,,
Comercio minorista,"2,608.00",21.38,NaN,NaN
Corporativo,984.00,33.03,NaN,NaN
Estándar urbano,NaN,NaN,"3,267.00",21.07
Express,NaN,NaN,"1,653.00",31.32
Programado,NaN,NaN,878.00,24.96
PyME,"2,206.00",24.61,NaN,NaN


**Resultado del KPI 2.**

| | Ticket enero (meta) | Ticket marzo | Índice marzo | Estado |
|---|---|---|---|---|
| **Total empresa** | $24,47 | $24,87 | 101,6 % | 🟢 |
| Azuay | $21,79 | $23,12 | 106,1 % | 🟢 |
| Tungurahua | $22,25 | $23,35 | 104,9 % | 🟢 |
| Pichincha | $26,05 | $26,86 | 103,1 % | 🟢 |
| Manabí | $22,88 | $22,59 | 98,8 % | 🟡 |
| Guayas | $26,89 | $26,44 | 98,3 % | 🟡 |
| Loja | $21,08 | $20,41 | 96,8 % | 🟡 |

**Interpretación.**
- **A nivel empresa, el ticket es estable** ($24,47, $24,38 y $24,87). Junto con el KPI 1, esto indica que el ingreso **crece por volumen y no por pedidos de mayor valor**: los pedidos por día pasaron de 59 en enero a 66–68 en febrero y marzo (+12 a +14 %).
- **Por provincia,** Azuay, Tungurahua y Pichincha suben su ticket. Guayas, Manabí y Loja quedan levemente bajo su nivel de enero, en zona de atención. Tungurahua tuvo una caída en febrero (94,5 %, crítico) y se recuperó en marzo.
- **Loja es la única provincia con señales negativas en ambos KPIs comerciales:** baja su cumplimiento de ventas en marzo (KPI 1) y su ticket es el más bajo de la empresa ($20,41).

**Nota metodológica y límites.** Las variaciones del ticket son pequeñas (entre 1 % y 6 %) y el valor de cada pedido varía bastante (desviación estándar de $10,74). Por eso, los estados amarillos son **señales a vigilar**, no alertas definitivas. Se reporta también la mediana, porque el promedio se eleva con pocos pedidos de alto valor (el máximo es $95).

**Implicación para la toma de decisiones.**
- **Comercial:** para crecer en valor, conviene revisar la mezcla de clientes y canales. Los pedidos corporativos y Express tienen tickets más altos ($33 y $31) que los de comercio minorista y estándar ($21).
- **Finanzas:** el ticket mide ingreso por pedido, no rentabilidad. Para evaluarla habría que incorporar los costos de entrega, que la base no incluye.

## 3.3 Indicador 3: Tasa de reclamos

In [18]:
kg.mostrar_ficha("k3")

Campo,Descripción
Nombre,KPI 3 – Tasa de reclamos
Objetivo,"Monitorear la calidad del servicio desde la experiencia del cliente, sin el efecto del volumen de pedidos."
Fórmula,"Σ reclamos_registrados ÷ N.º de pedidos × 100, con intervalo de confianza de Wilson al 95 %."
Fuente,"Base de pedidos LogiAndes S.A. (logiandes.csv), enero–marzo 2026: columnas reclamos_registrados, motivo_reclamo, pedido_id, fecha_pedido, provincia."
Frecuencia,"Mensual, con lectura acumulada trimestral para confirmar tendencias."
Meta,"≤ 3 reclamos por cada 100 pedidos, igual para todas las provincias (estándar de servicio)."
Umbral,🟢 ≤ 3 % | 🟡 3 % – 5 % | 🔴 > 5 %
Interpretación,Menor es mejor. Sobre 5 % indica un problema de servicio. Con pocos reclamos por mes la tasa varía por azar: confirmar con el intervalo de confianza y con la tasa trimestral antes de actuar.
Responsable,"Atención al cliente, con operaciones"


In [19]:
recl_mes = kg.tabla_mensual(df)
print(f"Tasa del trimestre: {kg.tabla_periodo(df).loc[0, 'tasa_reclamos']:.2f} %")
recl_mes[["mes", "pedidos", "reclamos", "k3", "k3_ic_inf", "k3_ic_sup", "estado_k3"]].round(2)

Tasa del trimestre: 3.23 %


,mes,pedidos,reclamos,k3,k3_ic_inf,k3_ic_sup,estado_k3
0,Enero,1837,68,3.70,2.93,4.67,amarillo
1,Febrero,1899,54,2.84,2.19,3.69,verde
2,Marzo,2062,65,3.15,2.48,4.00,amarillo


In [20]:
recl_trim = kg.tabla_periodo(df, "provincia")
kpi3_prov = prov_mes.pivot(index="provincia", columns="mes", values="k3")[["Enero", "Febrero", "Marzo"]]
kpi3_prov["Trimestre"] = recl_trim["tasa_reclamos"]
kpi3_prov["IC 95% trimestre"] = recl_trim.apply(lambda r: f"{r['k3_ic_inf']:.1f}–{r['k3_ic_sup']:.1f} %", axis=1)
kpi3_prov["Estado marzo"] = kpi3_prov["Marzo"].apply(lambda v: kg.etiqueta_estado(kg.estado("k3", v)))
kpi3_prov["Estado trimestre"] = recl_trim["estado_k3"].map(kg.etiqueta_estado)
kpi3_prov["Motivo principal"] = recl_trim["motivo_principal"]
kpi3_prov.sort_values("Trimestre", ascending=False).round(1)

mes,Enero,Febrero,Marzo,Trimestre,IC 95% trimestre,Estado marzo,Estado trimestre,Motivo principal
provincia,,,,,,,,
Manabí,8.40,3.60,7.70,6.60,5.0–8.7 %,🔴 Crítico,🔴 Crítico,Demora en entrega (19)
Loja,4.60,6.60,2.80,4.70,3.1–7.1 %,🟢 Cumple,🟡 Atención,Falta de actualización (10)
Guayas,4.50,3.70,4.20,4.10,3.3–5.2 %,🟡 Atención,🟡 Atención,Demora en entrega (28)
Pichincha,1.90,2.30,1.30,1.80,1.2–2.6 %,🟢 Cumple,🟢 Cumple,Demora en entrega (11)
Azuay,2.30,0.40,2.10,1.60,0.9–2.7 %,🟢 Cumple,🟢 Cumple,Paquete con incidencia (5)
Tungurahua,1.50,1.40,1.00,1.30,0.6–2.5 %,🟢 Cumple,🟢 Cumple,Paquete con incidencia (3)


In [21]:
kg.fig_kpi(df, "k3", mes=3, leyenda_abajo=False,
           titulo="<b>Manabí supera el umbral crítico de reclamos en marzo; la empresa queda cerca de la meta</b>").show()

In [22]:
# Caso Loja: el mes luce bien, pero el trimestre sigue fuera de meta
kg.fig_evolucion(df, "k3", "Loja").show()

**Resultado del KPI 3.**

| | Enero | Febrero | Marzo | Trimestre | Estado marzo | Motivo principal |
|---|---|---|---|---|---|---|
| **Total empresa** | 3,7 % | 2,8 % | 3,2 % | 3,2 % | 🟡 | — |
| Manabí | 8,4 % | 3,6 % | **7,7 %** | **6,6 %** | 🔴 | Demora en entrega |
| Loja | 4,6 % | 6,6 % | 2,8 % | 4,7 % | 🟢 (trimestre 🟡) | Falta de actualización |
| Guayas | 4,5 % | 3,7 % | 4,2 % | 4,1 % | 🟡 | Demora en entrega |
| Pichincha | 1,9 % | 2,3 % | 1,3 % | 1,8 % | 🟢 | Demora en entrega |
| Azuay | 2,3 % | 0,4 % | 2,1 % | 1,6 % | 🟢 | Paquete con incidencia |
| Tungurahua | 1,5 % | 1,4 % | 1,0 % | 1,3 % | 🟢 | Paquete con incidencia |

**Interpretación.**
- **A nivel empresa,** marzo (3,2 %) queda apenas sobre la meta de 3 %. Su margen de error (2,5 % a 4,0 %) incluye la meta, así que la empresa está **en el límite**, no ante un problema claro.
- **Manabí es el único caso crítico.** Registra 7,7 reclamos por cada 100 pedidos en marzo, más del doble de la meta, y su tasa del trimestre (6,6 %) confirma que no es un mes aislado. Todo su margen de error de marzo (5,0 % a 11,5 %) está sobre el umbral crítico. El motivo principal es la **demora en entrega**.
- **Guayas** se mantiene en zona amarilla los tres meses. Es un problema moderado pero constante, y por su volumen afecta al mayor número de clientes (71 reclamos en el trimestre).
- **Loja necesita una lectura cuidadosa.** En marzo aparece en verde (2,8 %), pero ese dato se basa en solo 4 reclamos y su tasa del trimestre (4,7 %) sigue en amarillo. El dashboard lo marca como **señal oculta**: el mes está en meta, pero el trimestre no.

**Nota metodológica y límites.** Con pocos reclamos por provincia y mes, la tasa mensual puede variar por azar. Por eso se muestran el margen de error y la tasa trimestral. El gráfico indica dónde hay más reclamos, pero no sus causas.

**Implicación para la toma de decisiones.**
- **Manabí es la prioridad de atención al cliente y operaciones.** Su motivo principal es la demora, y es la provincia con entregas más irregulares (KPI 4).
- **En Guayas,** por su volumen, una reducción moderada de la tasa tendría el mayor impacto en número de clientes.
- **En Loja,** el motivo principal es la falta de actualización del estado del pedido. Conviene mejorar la comunicación con el cliente, no solo acelerar las entregas.
- **Para el seguimiento,** una alerta se considera **confirmada** si se repite dos meses seguidos o si también aparece en la tasa trimestral. El dashboard aplica esta regla automáticamente.

## 3.4 Indicador 4: Mediana del tiempo de entrega

In [23]:
kg.mostrar_ficha("k4")

Campo,Descripción
Nombre,KPI 4 – Mediana del tiempo de entrega
Objetivo,Monitorear la rapidez de la entrega por provincia y mes frente al estándar operativo.
Fórmula,Mediana de tiempo_individual_entrega_min del mes. Complemento: RIC = Q3 − Q1 (variabilidad).
Fuente,"Base de pedidos LogiAndes S.A. (logiandes.csv), enero–marzo 2026: columnas tiempo_individual_entrega_min, canal_entrega, fecha_pedido, provincia."
Frecuencia,Mensual (seguimiento semanal en operaciones).
Meta,"Mediana ≤ 120 minutos, nivel que ya alcanzan Azuay y Tungurahua (exigente pero alcanzable)."
Umbral,🟢 ≤ 120 min | 🟡 120 – 150 min | 🔴 > 150 min
Interpretación,Menor es mejor. Se usa la mediana porque los tiempos tienen casos extremos. Sobre 150 min indica un problema operativo. El RIC muestra si las entregas son regulares o impredecibles.
Responsable,Operaciones y logística


In [24]:
ent_mes = kg.tabla_mensual(df)
print(f"Mediana del trimestre: {df[kg.T].median():.1f} min")
ent_mes[["mes", "k4", "ric_min", "pedidos", "estado_k4"]].round(1)

Mediana del trimestre: 129.3 min


,mes,k4,ric_min,pedidos,estado_k4
0,Enero,129.50,56.00,1837,amarillo
1,Febrero,127.80,55.90,1899,amarillo
2,Marzo,130.80,58.40,2062,amarillo


In [25]:
kpi4_prov = prov_mes.pivot(index="provincia", columns="mes", values="k4")[["Enero", "Febrero", "Marzo"]]
kpi4_prov["RIC marzo"] = prov_mes[prov_mes["mes_num"] == 3].set_index("provincia")["ric_min"]
kpi4_prov["Estado marzo"] = kpi4_prov["Marzo"].apply(lambda v: kg.etiqueta_estado(kg.estado("k4", v)))
kpi4_prov.sort_values("Marzo").round(1)

mes,Enero,Febrero,Marzo,RIC marzo,Estado marzo
provincia,,,,,
Tungurahua,114.60,108.30,110.90,48.30,🟢 Cumple
Azuay,116.80,113.90,114.10,41.10,🟢 Cumple
Pichincha,124.60,123.30,129.10,52.00,🟡 Atención
Guayas,136.70,132.80,138.00,61.00,🟡 Atención
Manabí,153.10,155.00,159.40,78.20,🔴 Crítico
Loja,159.40,169.00,166.20,63.90,🔴 Crítico


In [26]:
# ¿La diferencia se explica por la mezcla de canales? Mediana por provincia DENTRO de cada canal
mediana_canal = df.pivot_table(index="provincia", columns="canal_entrega", values=kg.T, aggfunc="median")
mediana_canal["Distancia mediana (km)"] = df.groupby("provincia")["distancia_km"].median()
mediana_canal["% Express"] = kg.tabla_periodo(df, "provincia")["pct_express"]
mediana_canal.sort_values("Estándar urbano").round(1)

canal_entrega,Estándar urbano,Express,Programado,Distancia mediana (km),% Express
provincia,,,,,
Tungurahua,114.00,66.60,135.90,6.60,23.20
Azuay,117.30,69.20,140.70,6.20,21.80
Pichincha,134.70,84.30,154.20,7.50,34.40
Guayas,144.10,97.30,166.10,8.60,34.20
Manabí,157.40,113.90,189.20,11.40,22.10
Loja,167.50,120.90,181.70,10.20,16.80


In [27]:
kg.fig_kpi(df, "k4", mes=3, leyenda_abajo=False,
           titulo="<b>Loja y Manabí superan el umbral crítico de entrega; solo Azuay y Tungurahua cumplen la meta</b>").show()

**Resultado del KPI 4.**

| | Enero | Febrero | Marzo | RIC marzo | Estado marzo |
|---|---|---|---|---|---|
| **Total empresa** | 130 min | 128 min | 131 min | 58 min | 🟡 |
| Tungurahua | 115 min | 108 min | 111 min | 48 min | 🟢 |
| Azuay | 117 min | 114 min | 114 min | 41 min | 🟢 |
| Pichincha | 125 min | 123 min | 129 min | 52 min | 🟡 |
| Guayas | 137 min | 133 min | 138 min | 61 min | 🟡 |
| Manabí | 153 min | 155 min | 159 min | 78 min | 🔴 |
| Loja | 159 min | 169 min | 166 min | 64 min | 🔴 |

**Interpretación.**
- **A nivel empresa,** la mediana se mantiene cerca de 130 minutos los tres meses, en zona amarilla y unos 10 minutos sobre la meta. No hay deterioro, pero tampoco mejora.
- **El patrón es consistente:** las mismas provincias quedan en las mismas zonas los tres meses. A diferencia del KPI 3, cada mediana se basa en cientos de pedidos.
- **Loja y Manabí están en rojo los tres meses.** Manabí además tiene la mayor variabilidad (RIC de 78 min): sus entregas son lentas e impredecibles.
- **Tungurahua y Azuay cumplen la meta,** lo que demuestra que entregar en menos de 2 horas es alcanzable dentro de la misma operación.

**¿La diferencia se explica por el canal?** Loja usa menos el canal Express (17 % de sus pedidos frente a 34 % en Pichincha y Guayas). Pero al comparar **dentro de un mismo canal**, Loja y Manabí siguen siendo las más lentas: en Estándar urbano tardan 168 y 157 minutos, frente a 114–117 en Tungurahua y Azuay. La mezcla de canales no es la explicación principal. Loja y Manabí sí tienen las mayores distancias medianas (10,2 y 11,4 km, frente a 6–7 km en las provincias más rápidas). Esto queda como **hipótesis**, que la sección 4.2 revisa por tramo de distancia.

**Nota metodológica y límites.** El análisis por canal y por distancia es exploratorio: indica posibles factores, pero no prueba causas. Otros factores que no están en la base (tráfico, número de repartidores, vías) también podrían influir.

**Implicación para la toma de decisiones.** Hay que priorizar la revisión de rutas, cobertura y capacidad de reparto en Loja y Manabí. En Manabí el reto es también hacer las entregas **más predecibles**. Las prácticas de Tungurahua y Azuay sirven como referencia interna.

## 3.5 Indicador 5: Porcentaje de entregas a tiempo

In [28]:
kg.mostrar_ficha("k5")

Campo,Descripción
Nombre,KPI 5 – Porcentaje de entregas a tiempo
Objetivo,Medir qué proporción de pedidos llega dentro del plazo máximo aceptable: lo que percibe cada cliente.
Fórmula,Pedidos con tiempo_individual_entrega_min ≤ 180 ÷ Total de pedidos del mes × 100 (IC de Wilson 95 %).
Fuente,"Base de pedidos LogiAndes S.A. (logiandes.csv), enero–marzo 2026: columnas tiempo_individual_entrega_min, pedido_id, fecha_pedido, provincia."
Frecuencia,Mensual (seguimiento semanal en operaciones).
Meta,≥ 90 % de pedidos en 180 min o menos (2 h = tiempo ideal del KPI 4; 3 h = máximo aceptable).
Umbral,"🟢 ≥ 90 % | 🟡 80 % – 89,9 % | 🔴 < 80 %"
Interpretación,Mayor es mejor. Complementa al KPI 4: la mediana muestra el tiempo típico y este KPI cuántos pedidos se salen del plazo. El plazo es único para todos los canales; Programado podría requerir otra ventana.
Responsable,"Operaciones, con seguimiento de atención al cliente"


In [29]:
at_mes = kg.tabla_mensual(df)
print(f"Entregas a tiempo en el trimestre: {df['a_tiempo'].mean() * 100:.1f} %")
at_mes[["mes", "pedidos", "a_tiempo", "fuera_plazo", "k5", "k5_ic_inf", "k5_ic_sup", "estado_k5"]].round(1)

Entregas a tiempo en el trimestre: 87.4 %


,mes,pedidos,a_tiempo,fuera_plazo,k5,k5_ic_inf,k5_ic_sup,estado_k5
0,Enero,1837,1604,233,87.30,85.70,88.80,amarillo
1,Febrero,1899,1670,229,87.90,86.40,89.30,amarillo
2,Marzo,2062,1791,271,86.90,85.30,88.20,amarillo


In [30]:
kpi5_prov = prov_mes.pivot(index="provincia", columns="mes", values="k5")[["Enero", "Febrero", "Marzo"]]
kpi5_prov["Fuera de plazo (marzo)"] = prov_mes[prov_mes["mes_num"] == 3].set_index("provincia")["fuera_plazo"]
kpi5_prov["Estado marzo"] = kpi5_prov["Marzo"].apply(lambda v: kg.etiqueta_estado(kg.estado("k5", v)))
kpi5_prov.sort_values("Marzo", ascending=False).round(1)

mes,Enero,Febrero,Marzo,Fuera de plazo (marzo),Estado marzo
provincia,,,,,
Azuay,99.20,99.20,98.90,3,🟢 Cumple
Tungurahua,96.00,99.50,96.20,8,🟢 Cumple
Pichincha,94.20,95.20,94.10,32,🟢 Cumple
Guayas,85.50,86.30,87.40,78,🟡 Atención
Manabí,66.70,72.20,64.00,94,🔴 Crítico
Loja,70.90,57.90,61.10,56,🔴 Crítico


In [31]:
kg.fig_kpi(df, "k5", mes=3, leyenda_abajo=False,
           titulo="<b>Loja y Manabí entregan a tiempo solo 6 de cada 10 pedidos, lejos de la meta del 90 %</b>").show()

In [32]:
# ¿Los pedidos fuera de plazo generan más reclamos?
display(kg.reclamos_segun_plazo(df).round(2))
kg.fig_reclamos_segun_plazo(df).show()

,pedidos,reclamos,tasa_reclamos,ic_inf,ic_sup
a_tiempo,,,,,
A tiempo (≤ 180 min),5065,144,2.84,2.42,3.34
Fuera de plazo (> 180 min),733,43,5.87,4.38,7.81


In [33]:
# Limitación del plazo único: cumplimiento por canal
kg.fig_histograma_plazo(df).show()
kg.tabla_periodo(df, "canal_entrega")[["pedidos", "pct_a_tiempo", "mediana_min"]].round(1)

,pedidos,pct_a_tiempo,mediana_min
canal_entrega,,,
Estándar urbano,3267,86.10,134.40
Express,1653,96.40,86.70
Programado,878,75.10,157.40


**Resultado del KPI 5.**

| | Enero | Febrero | Marzo | Fuera de plazo (marzo) | Estado marzo |
|---|---|---|---|---|---|
| **Total empresa** | 87,3 % | 87,9 % | 86,9 % | 271 pedidos | 🟡 |
| Azuay | 99,2 % | 99,2 % | 98,9 % | 3 | 🟢 |
| Tungurahua | 96,0 % | 99,5 % | 96,2 % | 8 | 🟢 |
| Pichincha | 94,2 % | 95,2 % | 94,1 % | 32 | 🟢 |
| Guayas | 85,5 % | 86,3 % | 87,4 % | 78 | 🟡 |
| Manabí | 66,7 % | 72,2 % | 64,0 % | 94 | 🔴 |
| Loja | 70,9 % | 57,9 % | 61,1 % | 56 | 🔴 |

**Interpretación.**
- **A nivel empresa,** cerca de 87 de cada 100 pedidos llegan dentro de las 3 horas, de forma estable, en zona amarilla y unos 3 puntos bajo la meta.
- **Loja y Manabí son críticas los tres meses.** En marzo entregaron a tiempo solo el 61 % y el 64 % de sus pedidos: **casi 4 de cada 10 clientes recibieron su pedido después de 3 horas**.
- **Dónde se concentra el problema:** de los 271 pedidos fuera de plazo en marzo, **228 (84 %) son de Manabí, Guayas y Loja**. Guayas, aunque está en amarillo, aporta 78 pedidos tardíos por su gran volumen.

**Relación con los reclamos (KPI 3).** Los pedidos fuera de plazo tienen una tasa de reclamos de **5,9 %** (IC 95 %: 4,4–7,8 %), frente a **2,8 %** (2,4–3,3 %) de los entregados a tiempo. Es más del doble y los intervalos no se superponen. Esto **sugiere** que cumplir el plazo está asociado a la satisfacción del cliente y convierte al KPI 5 en un **indicador adelantado** del KPI 3. Es una asociación, no una prueba de causa.

**Nota metodológica y límites.** El plazo de 180 minutos se aplica por igual a todos los canales. **Programado cumple solo el 75 %**, frente a 96 % de Express, porque su ventana de entrega acordada probablemente es distinta. El indicador podría penalizarlo de más. Con el horario comprometido de cada pedido, el KPI debería calcularse con el plazo de cada canal.

**Implicación para la toma de decisiones.** **Prioridad 1: Manabí y Loja**, en rojo en el KPI 4 y el KPI 5. Mejorar sus rutas y su capacidad de reparto tendría el mayor efecto en la experiencia del cliente y, por la asociación observada, también en los reclamos.

## 3.6 Tablero integrado de KPIs (marzo 2026)

Este tablero reúne los cinco indicadores en una sola vista. Es la base de la vista ejecutiva del dashboard: en lugar de cinco gráficos aislados, la gerencia ve **qué provincia está fuera de meta en qué KPI**.

In [34]:
kg.kpis_empresa(df, mes=3)[["KPI", "Indicador", "Valor", "Estado", "Meta"]]

,KPI,Indicador,Valor,Estado,Meta
kpi,,,,,
k1,KPI 1,Cumplimiento de meta de ventas mensuales,108.6 %,🟢 Cumple,🟢 ≥ 95 %
k2,KPI 2,Ticket promedio por pedido,101.6 %,🟢 Cumple,🟢 ≥ 100 %
k3,KPI 3,Tasa de reclamos,3.15 %,🟡 Atención,🟢 ≤ 3 %
k4,KPI 4,Mediana del tiempo de entrega,131 min,🟡 Atención,🟢 ≤ 120 min
k5,KPI 5,Porcentaje de entregas a tiempo,86.9 %,🟡 Atención,🟢 ≥ 90 %


In [35]:
kg.matriz_semaforo(df, mes=3)

,KPI 1 · Meta de ventas,KPI 2 · Ticket promedio,KPI 3 · Tasa de reclamos,KPI 4 · Mediana de entrega,KPI 5 · Entregas a tiempo,Pedidos
Total empresa,🟢 108.6 %,🟢 $24.87,🟡 3.15 %,🟡 131 min,🟡 86.9 %,2062
Manabí,🟢 109.1 %,🟡 $22.59,🔴 7.66 %,🔴 159 min,🔴 64.0 %,261
Loja,🟡 87.9 %,🟡 $20.41,🟢 2.78 %,🔴 166 min,🔴 61.1 %,144
Guayas,🟢 109.0 %,🟡 $26.44,🟡 4.21 %,🟡 138 min,🟡 87.4 %,618
Pichincha,🟢 114.8 %,🟢 $26.86,🟢 1.28 %,🟡 129 min,🟢 94.1 %,546
Azuay,🟢 109.9 %,🟢 $23.12,🟢 2.11 %,🟢 114 min,🟢 98.9 %,284
Tungurahua,🟢 103.4 %,🟢 $23.35,🟢 0.96 %,🟢 111 min,🟢 96.2 %,209


In [36]:
# Alertas priorizadas: confirmadas = se repiten el mes anterior o en el trimestre
kg.alertas(df, mes=3)[["Provincia", "KPI", "Indicador", "Valor", "Estado", "Lectura", "Confirmada", "Pedidos"]]

,Provincia,KPI,Indicador,Valor,Estado,Lectura,Confirmada,Pedidos
0,Manabí,KPI 3,Tasa de reclamos,7.66 %,🔴 Crítico,Mes evaluado,True,261
1,Manabí,KPI 4,Mediana del tiempo de entrega,159 min,🔴 Crítico,Mes evaluado,True,261
2,Manabí,KPI 5,Porcentaje de entregas a tiempo,64.0 %,🔴 Crítico,Mes evaluado,True,261
3,Loja,KPI 4,Mediana del tiempo de entrega,166 min,🔴 Crítico,Mes evaluado,True,144
4,Loja,KPI 5,Porcentaje de entregas a tiempo,61.1 %,🔴 Crítico,Mes evaluado,True,144
5,Guayas,KPI 2,Ticket promedio por pedido,98.3 %,🟡 Atención,Mes evaluado,True,618
6,Guayas,KPI 3,Tasa de reclamos,4.21 %,🟡 Atención,Mes evaluado,True,618
7,Guayas,KPI 4,Mediana del tiempo de entrega,138 min,🟡 Atención,Mes evaluado,True,618
8,Guayas,KPI 5,Porcentaje de entregas a tiempo,87.4 %,🟡 Atención,Mes evaluado,True,618
9,Pichincha,KPI 4,Mediana del tiempo de entrega,129 min,🟡 Atención,Mes evaluado,True,546


**Lectura del tablero.** **Manabí** acumula tres KPIs en rojo (reclamos, mediana y entregas a tiempo) y **Loja** dos (mediana y entregas a tiempo), además de la señal oculta en reclamos. **Guayas** tiene cuatro KPIs en amarillo: ninguno es crítico, pero afectan al mayor volumen de clientes. **Azuay y Tungurahua** cumplen los cinco KPIs. El problema de LogiAndes no es comercial (KPI 1 en verde); es **operativo y territorial**.

# 4. Estructura del dashboard

El prototipo (`app.py`) organiza la solución en **tres secciones obligatorias y una de gobernanza**. Todas comparten la barra lateral, de modo que la información es consistente al cambiar de pestaña.

```
┌──────────────────────────────────────────────────────────────────────────────┐
│ BARRA LATERAL             │ LogiAndes · Monitor de KPIs T1-2026              │
│  Vista para: [audiencia]  │ [Vista ejecutiva][Análisis territorial]          │
│  Mes evaluado: [Feb|Mar]  │ [Detalle operativo][KPIs y gobernanza]           │
│  Región / Provincia       │ ──────────────────────────────────────────────── │
│  Canal / Tipo de cliente  │ Pregunta clave de la audiencia                   │
│  [Restablecer filtros]    │ ┌KPI1┐┌KPI2┐┌KPI3┐┌KPI4┐┌KPI5┐ semáforo + Δ mes  │
│  N.º pedidos filtrados    │ Tablero provincia × KPI │ Alertas confirmadas    │
│  Fuente y periodo         │ Tendencia semanal       │ Matriz de priorización │
└──────────────────────────────────────────────────────────────────────────────┘
```

| Sección | Audiencia principal | Indicadores | Visualizaciones | Interactividad |
|---|---|---|---|---|
| **Vista ejecutiva** | Gerencia general, finanzas | KPI 1–5 con semáforo, variación mensual y lectura trimestral | Tarjetas KPI, tablero de semáforos provincia × KPI, alertas confirmadas, tendencia semanal, ventas por día, matriz de priorización | Selector de audiencia, mes evaluado, filtros globales, selector de métrica, tooltips |
| **Análisis territorial** | Gerencia, comercial | El KPI elegido por provincia frente a la meta | Gráfico del KPI (total empresa + provincias, mes anterior vs. mes evaluado); mapa de calor provincia × canal / tipo de cliente / tramo de distancia; *drill-down* de la provincia | Selector de KPI, selector de cruce, clic en una provincia para profundizar |
| **Detalle operativo** | Operaciones, atención al cliente | KPI 3, 4 y 5 por canal; motivos de reclamo | Histogramas por canal con el plazo de 180 min, boxplots, reclamos según plazo, Pareto de motivos, demanda día × hora, lista de pedidos | Filtros de excepción (con reclamo, fuera de plazo), búsqueda y descarga CSV |
| **KPIs y gobernanza** | Todas | Fichas técnicas | Fichas, verificación contra el Notebook, calidad de datos | Expansores por KPI |

**¿Por qué “mes evaluado” y no un rango de fechas libre?** Los KPIs 1 y 2 se calculan contra enero, y todos los KPIs tienen frecuencia mensual. Un rango de fechas arbitrario (por ejemplo, del 10 al 20 de febrero) produciría valores sin meta comparable. Limitar la selección a febrero o marzo evita lecturas incorrectas: es una forma de **controlar la sobreinteractividad**.

A continuación se generan los componentes del dashboard que no se mostraron en la sección 3, con las mismas funciones que usa `app.py`.

## 4.1 Vista ejecutiva: seguimiento semanal y priorización
Las fichas del KPI 4 y el KPI 5 proponen un seguimiento semanal en operaciones. En la vista semanal, las ventas se expresan **por día** para no castigar las semanas incompletas (1–4 de enero y 30–31 de marzo).

In [37]:
kg.fig_tendencia_semanal(df, "k3").show()
kg.fig_tendencia_semanal(df, "k5").show()

In [38]:
kg.fig_matriz_priorizacion(df).show()

**Lectura.** Las tasas semanales oscilan alrededor de la meta, sin tendencia clara al alza: el aumento de reclamos que percibe atención al cliente **no es general en el tiempo, se concentra en territorios**. La matriz de priorización (evolución de la dispersión del Taller 1) separa dos tipos de prioridad:
* **Guayas:** volumen alto y tasa en atención. Es la prioridad por **impacto**.
* **Manabí y Loja:** volumen menor y las tasas más altas. Es la prioridad por **calidad**.
* **Pichincha,** con mucho volumen y tasa baja, sirve como referencia para Guayas.

## 4.2 Análisis territorial: ¿geografía o desempeño?
El mapa de calor cruza el KPI 5 con el tramo de distancia para revisar la hipótesis del KPI 4.

In [39]:
kg.fig_heatmap(df, "k5", "tramo_distancia").show()
kg.fig_heatmap(df, "k3", "canal_entrega").show()

**Lectura.** La distancia influye: en todas las provincias el cumplimiento baja cuando la distancia aumenta. Pero **la brecha persiste dentro del mismo tramo**. En pedidos de 0 a 5 km, Manabí cumple el plazo en el 80 % de los casos y Loja en el 83 %, frente a 100 % en Azuay y 98 % en Pichincha. Las mayores distancias explican **parte** del problema; el resto apunta a factores operativos locales, que deben investigarse. En reclamos, el problema de Manabí y Guayas se concentra en el canal **Express**.

## 4.3 Detalle operativo

In [40]:
kg.fig_boxplot_entrega(df, "provincia").show()
kg.fig_pareto(df).show()
kg.fig_demanda_hora_dia(df).show()

**Lectura.** Los boxplots retoman el análisis de variabilidad del Taller 1, ahora con las líneas de la meta del KPI 4 (120 min) y del plazo del KPI 5 (180 min). **Demora en entrega (37 %) y falta de actualización (28 %)** suman el 65 % de los reclamos; ambos motivos se relacionan con el tiempo y con la comunicación del estado del pedido. La demanda se concentra entre las 10:00 y las 14:00, una referencia para dimensionar el personal de despacho y de atención.

# 5. Hallazgos y respuestas para la toma de decisiones

**¿El aumento de reclamos es un efecto normal del mayor volumen?** **No.** Al expresar los reclamos como tasa (KPI 3), Pichincha, con mucho volumen, cumple la meta, mientras que Manabí y Loja, con menos pedidos, tienen las tasas más altas. El volumen explica los reclamos **absolutos** de Guayas, pero no las diferencias en la tasa.

**¿Hay diferencias reales de calidad entre territorios?** **Sí, consistentes.** Manabí y Loja están en rojo en el KPI 4 y el KPI 5 los tres meses, y la brecha persiste dentro de un mismo canal y de un mismo tramo de distancia. El margen de error de Manabí en el KPI 3 está completamente sobre el umbral crítico.

**¿Hay dificultades propias de algunos canales?** **En parte.** Express concentra más reclamos en Manabí y Guayas. Programado parece incumplir el KPI 5, pero eso se debe al plazo único de 180 min (sección 3.5).

**¿Crecen las ventas?** **Sí**, aunque el crecimiento ocurrió en febrero (+14 % por día) y se sostuvo en marzo. Crece por volumen, no por valor del pedido (KPI 2 estable).

| Prioridad | Territorio | Señal (KPI) | Acción sugerida | Responsable | Seguimiento |
|---|---|---|---|---|---|
| 1 | Manabí | 🔴 KPI 3, KPI 4 y KPI 5 | Diagnóstico operativo: rutas, capacidad de flota y regularidad de entregas | Operaciones | KPI 4 y KPI 5 semanal |
| 2 | Loja | 🔴 KPI 4 y KPI 5; 🟡 KPI 1, KPI 2 y KPI 3 (trimestre) | Revisar cobertura para distancias de más de 10 km; mejorar la comunicación del estado del pedido | Operaciones y atención al cliente | KPI 5 por tramo de distancia; KPI 3 mensual |
| 3 | Guayas | 🟡 KPI 2, KPI 3, KPI 4 y KPI 5 | Reducir los pedidos fuera de plazo (78 en marzo) con las prácticas de Pichincha | Operaciones | KPI 5 |
| 4 | Toda la red | 28 % de reclamos por falta de actualización | Piloto de notificación proactiva del estado del pedido | Atención al cliente | KPI 3 por motivo |
| 5 | Comercial | KPI 2 estable | Estrategia de valor: clientes corporativos y canal Express | Comercial y finanzas | KPI 2 |

# 6. Coherencia entre el Notebook y el dashboard, y trazabilidad

1. **Una sola capa semántica:** el Notebook y `app.py` importan `kpis_grupo5.py`. No hay fórmulas duplicadas.
2. **Exportación de referencia:** el Notebook guarda en `outputs/` los KPIs mensuales de la empresa, los KPIs por provincia y la ficha técnica.
3. **Verificación automática:** la pestaña *KPIs y gobernanza* del dashboard recalcula los KPIs sin filtros y los compara con `outputs/kpis_referencia.json`. Si alguno difiere, muestra una alerta.
4. **Fidelidad con la versión anterior:** la prueba siguiente confirma que, con la meta fija, el módulo reproduce exactamente los resultados del primer cuaderno del grupo.

In [41]:
# Prueba 1: con la meta fija se obtienen los valores de la versión anterior del cuaderno
kg.AJUSTE_CALENDARIO = False
m = kg.tabla_mensual(df).set_index("mes")
assert round(m.loc["Febrero", "k1"], 1) == 98.1 and round(m.loc["Marzo", "k1"], 1) == 108.6
assert round(m.loc["Febrero", "k2"], 2) == 99.63 and round(m.loc["Marzo", "k2"], 2) == 101.63
assert round(m.loc["Enero", "k3"], 2) == 3.70 and round(m.loc["Marzo", "k5"], 1) == 86.9
kg.AJUSTE_CALENDARIO = True

# Prueba 2: los totales por provincia suman el total de la empresa
p = kg.tabla_mensual(df, "provincia")
assert p["pedidos"].sum() == len(df)
assert abs(p["ventas_usd"].sum() - df["ventas_asociadas_usd"].sum()) < 1e-6

# Prueba 3: un filtro calculado "a mano" coincide con el módulo
filtro = df[(df["region"] == "Costa") & (df["mes_num"] == 3)]
assert abs(kg.tabla_mensual(filtro).set_index("mes_num").loc[3, "k3"]
           - filtro["reclamos_registrados"].mean() * 100) < 1e-9
print("✔ Pruebas de coherencia superadas")

✔ Pruebas de coherencia superadas


In [42]:
salida = Path("outputs")
salida.mkdir(exist_ok=True)

mensual = kg.tabla_mensual(df).set_index("mes_num")
referencia = {
    "generado": pd.Timestamp.now().isoformat(timespec="seconds"),
    "fuente": kg.FUENTE,
    "registros": int(len(df)),
    "ajuste_calendario": kg.AJUSTE_CALENDARIO,
    "kpis": {k: {kg.MESES[m]: (None if pd.isna(v) else round(float(v), 6)) for m, v in mensual[k].items()}
             for k in kg.KPIS},
}
(salida / "kpis_referencia.json").write_text(json.dumps(referencia, ensure_ascii=False, indent=2), encoding="utf-8")
kg.tabla_mensual(df).to_csv(salida / "kpis_empresa_mensual.csv", index=False, encoding="utf-8-sig")
kg.tabla_mensual(df, "provincia").to_csv(salida / "kpis_provincia_mensual.csv", index=False, encoding="utf-8-sig")
kg.tabla_periodo(df, "provincia").to_csv(salida / "kpis_provincia_trimestre.csv", encoding="utf-8-sig")
kg.ficha_tecnica().to_csv(salida / "ficha_tecnica_kpis.csv", encoding="utf-8-sig")
print(json.dumps(referencia["kpis"], ensure_ascii=False, indent=1))
print("Archivos:", sorted(p.name for p in salida.iterdir()))

{
 "k1": {
  "Enero": null,
  "Febrero": 108.595787,
  "Marzo": 108.649125
 },
 "k2": {
  "Enero": null,
  "Febrero": 99.628319,
  "Marzo": 101.6333
 },
 "k3": {
  "Enero": 3.701688,
  "Febrero": 2.843602,
  "Marzo": 3.152279
 },
 "k4": {
  "Enero": 129.5,
  "Febrero": 127.8,
  "Marzo": 130.75
 },
 "k5": {
  "Enero": 87.316277,
  "Febrero": 87.941022,
  "Marzo": 86.85742
 }
}
Archivos: ['ficha_tecnica_kpis.csv', 'kpis_empresa_mensual.csv', 'kpis_provincia_mensual.csv', 'kpis_provincia_trimestre.csv', 'kpis_referencia.json']


# 7. Reflexión: riesgos y uso responsable

### 7.1 Riesgo de sobreinteractividad
Con 6 provincias, 3 canales, 3 tipos de cliente y 3 meses hay 162 combinaciones, muchas con pocos pedidos y menos de 5 reclamos. Demasiados filtros permiten **buscar hasta encontrar** una combinación que confirme una idea previa.
**Medidas aplicadas:**
* La vista ejecutiva se entiende sin interactuar.
* Los filtros están en un solo lugar y se muestran como resumen visible, con un botón para restablecerlos.
* El periodo se elige como **mes evaluado** y no como rango libre.
* Con menos de 30 pedidos, el dashboard muestra una **advertencia de muestra pequeña** y marca con * las celdas poco confiables.
* Las alertas se clasifican como **confirmadas** solo si se repiten.

### 7.2 Sesgos visuales
* **Efecto calendario:** comparar totales de meses con distinto número de días exageraba el crecimiento de marzo. Se corrigió con la meta prorrateada y las ventas por día.
* **Doble eje:** se evita. Ventas, reclamos y tiempos se muestran en gráficos separados.
* **Ejes truncados:** las barras empiezan en cero. Las líneas de meta y de umbral dan contexto sin exagerar diferencias pequeñas.
* **Mapas coropléticos:** se descartan porque el área de cada provincia distorsiona la comparación (continuidad del Taller 1).
* **Falsa precisión:** el margen de error en el KPI 3 y el KPI 5 evita sobreinterpretar diferencias de uno o dos puntos.
* **Color:** el semáforo siempre lleva texto (Cumple, Atención, Crítico), y el total de la empresa usa azul para no confundirse con un estado.

### 7.3 Privacidad
La base no contiene datos personales directos: no hay nombres, cédulas, direcciones ni teléfonos, y `pedido_id` es un **seudónimo**. Aun así, la combinación de fecha y hora exacta, provincia, tipo de cliente y monto podría **reidentificar** a un cliente corporativo en una provincia pequeña. Por los principios de minimización y finalidad de la **Ley Orgánica de Protección de Datos Personales del Ecuador (LOPDP, 2021)**:
* La lista de pedidos solo aparece en el detalle operativo, para uso interno.
* En producción, el dashboard debería requerir autenticación y control de acceso por rol.
* Si se incorporan datos del cliente (nombre o RUC), deben enmascararse.

### 7.4 Gobernanza
* **Responsables de cada KPI:** KPI 1 Gerencia comercial; KPI 2 Comercial y finanzas; KPI 3 Atención al cliente; KPI 4 y KPI 5 Operaciones. BI administra el cálculo, no las metas.
* **Definición única:** la ficha técnica vive en el código (`kg.KPIS`) y se muestra en el dashboard. Cambiar una meta es un cambio documentado, no una edición manual de un gráfico.
* **Supuestos explícitos:** el crecimiento de 5 %, el plazo de 180 min, las metas y el ajuste por calendario son propuestas académicas que debe aprobar un comité de indicadores.
* **Calidad y linaje:** los controles se ejecutan al cargar los datos, y la fuente, el periodo y la fecha de generación son visibles.

### 7.5 Límites interpretativos
* **Asociación no es causalidad.** Que los pedidos fuera de plazo tengan más reclamos, o que Loja y Manabí recorran más distancia, no prueba la causa. Hay variables no observadas, como la flota, el personal, las vías y el clima.
* **Horizonte corto.** Con tres meses no se puede separar la tendencia de la estacionalidad. Enero, la línea base, podría ser un mes atípico.
* **Eventos poco frecuentes.** Con 187 reclamos en total, las tasas mensuales por provincia son inestables. Un 0 % o un 1 % en un mes no significa servicio perfecto.
* **Reclamo no es lo mismo que insatisfacción.** Solo se observa al cliente que reclama.
* **Metas propuestas.** El color del semáforo depende de las metas. Si cambian, cambia el color, no el desempeño.

# 8. Conclusiones y ejecución

1. **LogiAndes crece comercialmente** (KPI 1 en verde: 108,6 % en febrero y marzo con la meta ajustada), pero el crecimiento es **por volumen**, no por valor del pedido (KPI 2 estable).
2. **El problema es operativo y territorial.** Manabí y Loja están en rojo en el tiempo de entrega y en las entregas a tiempo; Guayas, en atención en cuatro KPIs y con el mayor número de clientes afectados.
3. **Entregar a tiempo está asociado a menos reclamos** (2,8 % frente a 5,9 %). El KPI 5 funciona como **indicador adelantado** del KPI 3.
4. Los gráficos del Taller 1 se integraron en un **producto BI monitoreable**: cinco KPIs con ficha técnica, semáforo, alertas confirmadas, tres secciones por audiencia, filtros comunes, *drill-down* y verificación automática de coherencia con este Notebook.

### Anexo: cambios frente a la versión anterior del cuaderno
| Cambio | Motivo |
|---|---|
| Meta del KPI 1 prorrateada por días del mes | Febrero (28 días) quedaba penalizado; el “+10,8 % de marzo” era un efecto calendario |
| Corrección sobre Loja en el KPI 1 | Su caída es de marzo, no una tendencia: por día subió en febrero |
| Un solo módulo para los cinco KPIs y gráficos | Elimina código duplicado y garantiza la coherencia con el dashboard |
| Lectura trimestral y alertas “confirmadas” | Aplica la regla del grupo: no reaccionar ante variaciones aleatorias |
| Tablero integrado provincia × KPI | De cinco gráficos aislados a una vista de decisión |
| Análisis por tramo de distancia | Pone a prueba la hipótesis de la distancia del KPI 4 |
| Secciones de audiencias, estructura del dashboard, coherencia y reflexión | Requisitos del reto |

### Cómo ejecutar
```bash
cd Taller_2_Grupo5
source ~/venvs/reto_logiandes/bin/activate      # o crear un entorno: pip install -r requirements.txt
jupyter nbconvert --to notebook --execute --inplace Taller_2_Grupo5.ipynb   # regenera outputs/
streamlit run app.py
```